# Data Cleaning

First of all we load the data and make sure it is loaded correctly by previewing few items.

In [ ]:
import pandas as pd

file_path = "\Downloads\US_Accidents_50k.xlsx"
df = pd.read_excel(file_path, skiprows=1)

df.head(10)

: 

Now Let's check for douplicate items and remove them

In [42]:
duplicates = df[df.duplicated()]
num_duplicates = duplicates.shape[0]
if num_duplicates == 0:
    print("✅ No duplicate rows found. Proceeding to the next step...")
else:
    before_size = df.shape[0]
    # Remove duplicates (keep first occurrence)
    df = df.drop_duplicates()
    after_size = df.shape[0]
    print(f"⚠️ {num_duplicates} duplicate rows were removed.")
    print(f"   Data size reduced from {before_size} to {after_size}.")


✅ No duplicate rows found. Proceeding to the next step...


Now let's handle missing data with the following strategy:
- For numeric columns: replace missing values with the column’s median.
- For categorical/text columns: replace missing values with the most frequent (mode) value.

In [43]:
for col in df.columns:
    if df[col].isnull().any():  # Only process columns with missing data
        if pd.api.types.is_numeric_dtype(df[col]):
            median_value = df[col].median()
            df[col].fillna(median_value, inplace=True)
            print(f"   - Filled numeric column '{col}' with median value: {median_value}")
        else:
            mode_value = df[col].mode()[0]
            df[col].fillna(mode_value, inplace=True)
            print(f"   - Filled categorical column '{col}' with most frequent value: {mode_value}")

print("\n✅ Missing data handling completed.")

   - Filled numeric column 'End_Lat' with median value: 36.150102000000004
   - Filled numeric column 'End_Lng' with median value: -87.9976635
   - Filled categorical column 'Street' with most frequent value: I-95 N
   - Filled categorical column 'Timezone' with most frequent value: US/Eastern
   - Filled categorical column 'Airport_Code' with most frequent value: KCQT
   - Filled categorical column 'Weather_Timestamp' with most frequent value: 2018-01-24 08:53:00
   - Filled numeric column 'Temperature(F)' with median value: 64.0
   - Filled numeric column 'Wind_Chill(F)' with median value: 62.0
   - Filled numeric column 'Humidity(%)' with median value: 68.0
   - Filled numeric column 'Pressure(in)' with median value: 29.85
   - Filled numeric column 'Visibility(mi)' with median value: 10.0
   - Filled categorical column 'Wind_Direction' with most frequent value: CALM
   - Filled numeric column 'Wind_Speed(mph)' with median value: 7.0
   - Filled numeric column 'Precipitation(in)' wi

/tmp/ipython-input-2501780541.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_value, inplace=True)
/tmp/ipython-input-2501780541.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd


Now let's handle outliers in the numeric columns using the Interquartile Range (IQR) method:
- **Strategy**: Identify extreme values that fall outside the range of $Q1 - 5.0 \times \text{IQR}$ and $Q3 + 5.0 \times \text{IQR}$.
- **Action**: Remove the rows containing these extreme outlier values to clean the distribution and improve model performance.

In [44]:
import numpy as np

print("Starting outlier detection and removal (Updated 5.0 x IQR method)...")
df_clean = df.copy()
original_size = df_clean.shape[0]
rows_removed = 0

# The multiplier to define the outlier boundary (increased to 5.0)
IQR_MULTIPLIER = 5.0

# Identify numeric columns (excluding 'ID' and 'Severity')
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
cols_to_exclude = ['ID', 'Severity']
numeric_cols = [col for col in numeric_cols if col not in cols_to_exclude]

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    # Define outlier boundaries using the new, wider multiplier
    lower_bound = Q1 - IQR_MULTIPLIER * IQR
    upper_bound = Q3 + IQR_MULTIPLIER * IQR

    # Create a boolean mask for non-outlier data
    non_outlier_mask = (df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)

    # Identify outliers in the current column
    outliers = df_clean[~non_outlier_mask]

    if len(outliers) > 0:
        # Remove the identified outlier rows
        df_clean = df_clean.loc[non_outlier_mask]

        removed_count = len(outliers)
        rows_removed += removed_count
        print(f"   - Removed {removed_count} outlier rows from column '{col}' (5.0 x IQR).")

final_size = df_clean.shape[0]

if rows_removed > 0:
    print(f"\n✅ Outlier removal completed. Total rows removed: {rows_removed}.")
    print(f"   Data size reduced from {original_size} to {final_size}.")
    df = df_clean # Update the main DataFrame with the cleaned version
else:
    print("\n✅ No extreme outliers found in the numeric columns (or none were removed).")

Starting outlier detection and removal (Updated 5.0 x IQR method)...
   - Removed 13 outlier rows from column 'End_Lat' (5.0 x IQR).
   - Removed 494 outlier rows from column 'Distance(mi)' (5.0 x IQR).
   - Removed 2 outlier rows from column 'Wind_Chill(F)' (5.0 x IQR).
   - Removed 262 outlier rows from column 'Pressure(in)' (5.0 x IQR).
   - Removed 1820 outlier rows from column 'Visibility(mi)' (5.0 x IQR).
   - Removed 8 outlier rows from column 'Wind_Speed(mph)' (5.0 x IQR).
   - Removed 191 outlier rows from column 'Precipitation(in)' (5.0 x IQR).

✅ Outlier removal completed. Total rows removed: 2790.
   Data size reduced from 9998 to 7208.


Now let's perform data formatting and standardization on all numeric and geographical columns:
- **Strategy**: Ensure all numeric columns contain only digits and a decimal point, and apply precise decimal place formatting for consistency.
- **Action**:
    * **General Numerics**: Remove any non-numeric characters (except the minus sign) and convert columns back to the `float` type.
    * **Geographical Co-ordinates** (`Start_Lat`, `Start_Lng`, `End_Lat`, `End_Lng`): Format as strings with **6 decimal places** (e.g., "29.698118").
    * **Weather/Wind/Pressure** (`Temperature(F)`, `Wind_Chill(F)`, `Humidity(%)`, `Pressure(in)`, `Visibility(mi)`, `Wind_Speed(mph)`, `Precipitation(in)`): Format as strings with **2 decimal places** (e.g., "00.00").
    * **Distance** (`Distance(mi)`): Format as a string with **3 decimal places** (e.g., "0.056").

In [45]:
import numpy as np

print("Starting numeric column formatting and standardization...")

# --- 1. General Numeric Cleaning (Removing non-numeric characters) ---
print("\n1. Cleaning all numeric columns...")

# Identify all numeric columns for general cleaning
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
cols_to_exclude = ['ID', 'Severity']
cols_for_cleaning = [col for col in numeric_cols if col not in cols_to_exclude]

for col in cols_for_cleaning:
    # Convert column to string, remove commas (if any), and force to float
    # Note: Since the previous steps ensured these are already numeric (by median/mode imputation),
    # this step primarily ensures they are stored as floating-point numbers.
    try:
        df[col] = df[col].astype(str).str.replace(r'[^\d\.\-]', '', regex=True).astype(float)
        # print(f"   - Cleaned column '{col}' to ensure it's purely numeric.")
    except Exception as e:
        print(f"   - ⚠️ Could not clean column '{col}'. Error: {e}")

# --- 2. Applying Specific Decimal Formats ---
print("\n2. Applying specific decimal formats to columns...")

# A helper function to apply format without changing the underlying numeric value
def apply_format_string(series, format_string):
    """Converts numeric series to string with specified format."""
    # We use round() for precision and then format it as a string
    return series.round(len(format_string.split('.')[-1])).apply(lambda x: format_string.format(x))

# A dictionary mapping columns to their required format strings
format_specs = {
    # 6 decimal points for Lat/Lng
    'Start_Lat': "{:.6f}",
    'Start_Lng': "{:.6f}",
    'End_Lat': "{:.6f}",
    'End_Lng': "{:.6f}",

    # 2 decimal points for weather/wind/pressure
    'Temperature(F)': "{:.2f}",
    'Wind_Chill(F)': "{:.2f}",
    'Humidity(%)': "{:.2f}",
    'Pressure(in)': "{:.2f}",
    'Visibility(mi)': "{:.2f}",
    'Wind_Speed(mph)': "{:.2f}",
    'Precipitation(in)': "{:.2f}",

    # 3 decimal points for Distance(mi)
    'Distance(mi)': "{:.3f}"
}

# Apply formatting
for col, fmt in format_specs.items():
    if col in df.columns:
        # Convert the numeric column to a formatted string
        df[col] = df[col].apply(lambda x: fmt.format(x) if pd.notna(x) else x)
        print(f"   - Formatted column '{col}' to match string pattern {fmt.replace('{:', '').replace('}', '')}.")
    else:
        print(f"   - Column '{col}' not found in DataFrame. Skipping formatting.")

print("\n✅ Formatting and standardization completed.")

Starting numeric column formatting and standardization...

1. Cleaning all numeric columns...

2. Applying specific decimal formats to columns...
   - Formatted column 'Start_Lat' to match string pattern .6f.
   - Formatted column 'Start_Lng' to match string pattern .6f.
   - Formatted column 'End_Lat' to match string pattern .6f.
   - Formatted column 'End_Lng' to match string pattern .6f.
   - Formatted column 'Temperature(F)' to match string pattern .2f.
   - Formatted column 'Wind_Chill(F)' to match string pattern .2f.
   - Formatted column 'Humidity(%)' to match string pattern .2f.
   - Formatted column 'Pressure(in)' to match string pattern .2f.
   - Formatted column 'Visibility(mi)' to match string pattern .2f.
   - Formatted column 'Wind_Speed(mph)' to match string pattern .2f.
   - Formatted column 'Precipitation(in)' to match string pattern .2f.
   - Formatted column 'Distance(mi)' to match string pattern .3f.

✅ Formatting and standardization completed.


The final step is to save the cleaned and standardized dataset to a new Excel file:
- **Strategy**: Use the pandas `to_excel` function to save the final DataFrame.
- **Action**: Append the suffix "_cleaned" to the original filename to distinguish the processed file from the raw input.

In [46]:
import os

# Assume 'df' is your final, cleaned DataFrame

# 1. Define the original file name and directory
# Based on the file_path in your cell 19: "/content/US_Accidents_March23_sampled_10k.xlsx"
original_file_path = "/content/US_Accidents_March23_sampled_10k.xlsx"

# 2. Construct the new cleaned filename
base_name = os.path.basename(original_file_path)
file_name_without_ext, extension = os.path.splitext(base_name)
new_file_name = f"{file_name_without_ext}_cleaned{extension}"

# 3. Define the full path for the new file (saving it in the same directory)
output_file_path = os.path.join(os.path.dirname(original_file_path), new_file_name)

# 4. Save the DataFrame to the new Excel file
# index=False prevents pandas from writing the DataFrame index as a column
try:
    df.to_excel(output_file_path, index=False)
    print(f"\n✅ Successfully saved the cleaned data to:")
    print(f"   {output_file_path}")
    print(f"   Final DataFrame shape: {df.shape}")
except Exception as e:
    print(f"\n❌ Error during file saving: {e}")


✅ Successfully saved the cleaned data to:
   /content/US_Accidents_March23_sampled_10k_cleaned.xlsx
   Final DataFrame shape: (7208, 46)
